# Classifieur de compétences IA (v2, 20 labels)

Charge un modèle CamemBERT fine-tuné et prédit les compétences IA d'une formation.

**Prérequis** : avoir exécuté `make ia-train` ou posséder un checkpoint dans `models/ia-classifier-v2/final`.

In [5]:
from __future__ import annotations
import sys
from pathlib import Path

# S'assurer que le projet est dans le path
sys.path.insert(0, str(Path.cwd()))

from src.models.ia_classifier import IAClassifier

ModuleNotFoundError: No module named 'src'

In [ ]:
# Charger le modele
# Si vous avez un checkpoint entraine ailleurs, remplacez le chemin
MODEL_DIR = "models/ia-classifier-v2/final"

if Path(MODEL_DIR).exists():
    clf = IAClassifier(MODEL_DIR, top_k=5)
    print(f"Modele charge : {clf.num_labels} labels, device={clf.device}")
else:
    print(f"Checkpoint introuvable : {MODEL_DIR}")
    print("Lancez d'abord : make ia-train")

In [ ]:
# Exemple de prediction
exemples = [
    "Formation approfondie en Machine Learning et Deep Learning avec TensorFlow et PyTorch",
    "Initiation a Python pour l'analyse de donnees et la visualisation",
    "Decouvrez le Prompt Engineering et l'IA Generative avec LangChain",
    "Formation sur l'ethique de l'IA et la gestion de projet IA",
    "SQL avance et Big Data : architectures data engineering",
]

for texte in exemples:
    preds = clf.predict(texte)[0]
    retenus = [p for p in preds if p["selected"]]
    print(f"\n{'─'*60}")
    print(f"TEXTE : {texte[:80]}..." if len(texte) > 80 else f"TEXTE : {texte}")
    if retenus:
        print(f"COMPETENCES ({len(retenus)}) :")
        for p in retenus:
            print(f"  - {p['label']:35s} {p['probability']:.1%}  (seuil: {p['threshold']:.2f})")
    else:
        print("Aucune competence retenue (scores sous les seuils)")

In [ ]:
# Top-3 pour chaque exemple
print("Top 3 predictions par formation :")
for texte in exemples:
    preds = clf.predict(texte, top_k=3)[0]  # top_k override
    labels = [p["label"] for p in preds]
    probs = [f"{p['probability']:.0%}" for p in preds]
    print(f"  {texte[:50]:50s} → {', '.join(f'{l} ({p})' for l,p in zip(labels, probs))}")

In [ ]:
# Prediction par label et probabilite (trie)
preds = clf.predict(exemples[0])[0]
print(f"\nTous les labels pour : {exemples[0][:60]}...")
print(f"{'Label':35s} {'Prob':>8s} {'Seuil':>6s} {'Retenu':>8s}")
print("-"*60)
for p in preds:
    print(f"{p['label']:35s} {p['probability']:7.1%}  {p['threshold']:.2f}  {'OUI' if p['selected'] else 'non':>8s}")

In [ ]:
# Prediction batch
textes = [
    "Computer Vision et NLP pour l'analyse de documents",
    "Formation No-code / Low-code : creez des apps sans coder",
]
resultats = clf.predict_labels(textes)
for texte, labels in zip(textes, resultats):
    print(f"{texte[:50]:50s} → {labels}")

In [ ]:
# Matrice des probabilites brutes
probas = clf.predict_probas(exemples)
print(f"Shape : {probas.shape}  (batch x {clf.num_labels} labels)")
print(f"Labels : {clf.labels}")

---
### Configuration

Le classifieur utilise :
- **Modèle** : CamemBERT-base avec tête de classification multilabel
- **Seuils** : optimisés par label sur le split de validation (fichier `thresholds.json`)
- **Fallback** : seuil global de 0.50 si `thresholds.json` absent
- **Taxonomie** : 20 labels définis dans `config/ia_taxonomy_v2.json`
- **normalisation** : alias map intégré (`"ethique ia"` → `"Ethique IA & RGPD"`, etc.)